<a href="https://colab.research.google.com/github/Maee127/Adversarial-Notebooks/blob/master/%236/Notebook_6_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================
# Adversarial Attacks Series -- Note 06 (REVISED)
# Architectural Awareness: Building the Sensor into the Boat
# =============================================================
#
# Series:  Humble Model / Architectural Awareness
# Dataset: CIFAR-10
# Model:   CNN (same backbone as Essay #5)
#
#   Notebook structure:
#   Part A: Imports and Setup
#   Part B: Dataset Loading (CIFAR-10)
#   Part C: Model Architecture (shared backbone)
#   Part D: Train or load all three models
#   Part E: Attacks
#   Part F: Signal collection
#   Part G: Calibrate thresholds ON CALIB DATA ONLY
#   Part H: Evaluate each signal as a standalone gate
#   Part I: Fused gate
#   Part J: Summary table
# =============================================================
#
# CHANGES IN THIS REVISION (see chat for the full explanation of each):
#   1. RISK FORMULA FIXED. The previous version computed:
#        risk = 100 * ((total - correct) + deferred) / (total + deferred)
#      which counts every deferred sample as a failure -- a PERFECT
#      gate that defers exactly its own errors would still score
#      risk == deferral_rate under that formula, not ~0%. That
#      inverts this series' entire premise that deferral is the SAFE
#      choice. Risk is now `100 - accuracy(on predicted)`, matching
#      Essays #4 and #5 exactly, so numbers are comparable across the
#      series. Deferral rate and coverage are reported alongside it
#      as separate numbers, the way #4/#5 did -- not folded into risk.
#   2. clamp_valid() used in EVERY attack function. The previous
#      version used torch.clamp(x, 0, 1) in most of them, which is
#      wrong for CIFAR-normalized inputs -- exactly the bug Essay #5
#      fixed, that didn't carry over to this notebook.
#   3. The calibration split is now actually used. Every threshold
#      (confidence, disagreement, evidence) is calculated on
#      calib_loader and applied to eval_loader -- not calculated
#      in-sample on the same data being evaluated.
#   4. Multi-head's threshold is now calibrated (75th percentile of
#      disagreement on the calibration set), not a hardcoded 0.05
#      that never fired.
#   5. Boundary distance is computed for every image actually used in
#      the fused-gate sample -- no more padding unsampled images in a
#      batch with a copy of an unrelated image's score. Sample size
#      is controlled by evaluating fewer total images, not by
#      silently faking values within a batch.
#   6. Part H now includes a STANDALONE boundary-distance row, so its
#      individual contribution can be compared to confidence,
#      disagreement, and evidence before looking at the fused blend.
#   7. Fused gate uses Essay #5's OR-based fusion (defer if ANY
#      individual signal crosses ITS OWN calibrated threshold) instead
#      of a weighted linear blend of differently-scaled raw values.
#      The previous weighted-sum design mixed confidence (0-1, higher
#      = safer), disagreement (unbounded, higher = riskier), evidence
#      (unbounded, higher = safer), and boundary distance (0-1ish,
#      LOWER = riskier) into one number without a principled way to
#      make their scales comparable. OR-based fusion sidesteps that:
#      each signal only needs to be individually well-calibrated.
# =============================================================


In [1]:
# -------------------------------------------------------------
# Part A: Imports and Setup
# -------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import random

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


Using device: cuda


In [2]:
# -------------------------------------------------------------
# Part B: Dataset Loading (CIFAR-10) with held-out split
# -------------------------------------------------------------

cifar_transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform_test)

calib_size = int(0.2 * len(test_dataset))
eval_size = len(test_dataset) - calib_size
calib_dataset, eval_dataset = random_split(test_dataset, [calib_size, eval_size],
                                            generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset):,}")
print(f"Calibration samples: {len(calib_dataset):,}  <- thresholds come from here now")
print(f"Evaluation samples: {len(eval_dataset):,}  <- final numbers come from here")

CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
CIFAR_MIN = ((0 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)
CIFAR_MAX = ((1 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)


100%|██████████| 170M/170M [22:41<00:00, 125kB/s]


Training samples: 50,000
Calibration samples: 2,000  <- thresholds come from here now
Evaluation samples: 8,000  <- final numbers come from here


In [3]:
# -------------------------------------------------------------
# Part C: Model Architecture (Shared Backbone)
# -------------------------------------------------------------

class BackboneCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x)); x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x)); x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return x

class StandardCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = BackboneCNN()
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, x):
        return self.fc_out(self.backbone(x))

class MultiHeadCNN(nn.Module):
    def __init__(self, backbone, num_heads=5, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleList([nn.Linear(256, num_classes) for _ in range(num_heads)])

    def forward(self, x):
        features = self.backbone(x)
        return torch.stack([head(features) for head in self.heads], dim=1)

    def forward_with_disagreement(self, x):
        logits = self.forward(x)
        probs = F.softmax(logits, dim=2)
        return probs.mean(dim=1), probs.var(dim=1)

class EvidentialCNN(nn.Module):
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.fc_alpha = nn.Linear(256, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return F.softplus(self.fc_alpha(features)) + 1.0

    def forward_with_evidence(self, x):
        alpha = self.forward(x)
        S = alpha.sum(dim=1, keepdim=True)
        return alpha / S, S.squeeze(1)

def evidential_loss(alpha, labels_onehot):
    S = alpha.sum(dim=1, keepdim=True)
    return (labels_onehot * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=1).mean()


In [4]:
# -------------------------------------------------------------
# Part D: Train or load all three models -- training loop unchanged,
# only the checkpoint filenames matter. If you already have
# checkpoints from the previous run, they're still valid (training
# wasn't buggy, only evaluation/attacks were) -- just point at them.
# -------------------------------------------------------------

def train_standard(model, loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
    return model

def train_multihead(model, loader, epochs=20, num_heads=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = torch.stack([criterion(logits[:, h, :], labels) for h in range(num_heads)]).mean()
            loss.backward()
            optimizer.step()
    return model

def train_evidential(model, loader, epochs=20):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            alpha = model(images)
            loss = evidential_loss(alpha, F.one_hot(labels, num_classes=10).float())
            loss.backward()
            optimizer.step()
    return model

def load_or_train(model, path, train_fn, loader):
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded {path}")
    else:
        print(f"Training -> {path}")
        model = train_fn(model, loader)
        torch.save({'model_state_dict': model.state_dict()}, path)
    return model

baseline_model = load_or_train(StandardCNN().to(DEVICE), 'checkpoint_baseline_cifar10.pth', train_standard, train_loader)
multi_head_model = load_or_train(MultiHeadCNN(BackboneCNN(), num_heads=5).to(DEVICE), 'checkpoint_multihead.pth', train_multihead, train_loader)
evidential_model = load_or_train(EvidentialCNN(BackboneCNN()).to(DEVICE), 'checkpoint_evidential.pth', train_evidential, train_loader)

for m in [baseline_model, multi_head_model, evidential_model]:
    m.eval()


Training -> checkpoint_baseline_cifar10.pth


Epoch 19: 100%|██████████| 782/782 [00:21<00:00, 36.63it/s]


Training -> checkpoint_multihead.pth


Epoch 19: 100%|██████████| 782/782 [00:22<00:00, 34.67it/s]


Training -> checkpoint_evidential.pth


Epoch 19: 100%|██████████| 782/782 [00:22<00:00, 35.27it/s]


In [5]:
# -------------------------------------------------------------
# Part E: Attacks -- ONE set of correct attack functions, reused by
# everything below. clamp_valid() everywhere, no exceptions.
# -------------------------------------------------------------

def fgsm_standard(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    loss = nn.CrossEntropyLoss()(model(images), labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_standard(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        loss = nn.CrossEntropyLoss()(model(images_adv), labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def _multihead_loss(model, images, labels):
    logits = model(images)
    criterion = nn.CrossEntropyLoss()
    return torch.stack([criterion(logits[:, h, :], labels) for h in range(logits.size(1))]).mean()

def fgsm_multihead(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    loss = _multihead_loss(model, images, labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_multihead(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        loss = _multihead_loss(model, images_adv, labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def fgsm_evidential(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    alpha = model(images)
    loss = nn.CrossEntropyLoss()(alpha, labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_evidential(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        alpha = model(images_adv)
        loss = nn.CrossEntropyLoss()(alpha, labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def estimate_boundary_distance(model, image, label, max_iters=30):
    model.eval()
    image = image.clone().detach().requires_grad_(True)
    loss = nn.CrossEntropyLoss()(model(image), torch.tensor([label], device=DEVICE))
    model.zero_grad(); loss.backward()
    grad = image.grad.data.sign()
    low, high = 0.0, 1.0
    for _ in range(max_iters):
        mid = (low + high) / 2
        perturbed = clamp_valid(image + mid * grad)
        with torch.no_grad():
            pred = model(perturbed).argmax().item()
        if pred != label:
            high = mid
        else:
            low = mid
    return (low + high) / 2

PGD_PARAMS = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}


In [6]:
# -------------------------------------------------------------
# Part F: Signal collection -- confidence, disagreement, evidence,
# boundary distance, all computed the SAME way on calib vs eval,
# clean vs adversarial (PGD). Boundary distance is limited by
# n_boundary_samples (a real subsample, no padding).
# -------------------------------------------------------------

def collect_signals(baseline, multihead, evidential, loader, condition='clean',
                     n_samples=None, n_boundary_samples=300, boundary_seed=42):
    """
    condition: 'clean' or 'adversarial' (PGD against baseline_model).
    n_samples: cap total images processed (None = whole loader).
    n_boundary_samples: how many images get a real boundary-distance
      score (expensive: up to 30 passes each). Images NOT selected get
      boundary=None -- NOT padded with a stale value.

    FIX: which images get a boundary score is now a RANDOM subset of
    the whole pass, not just "the first N in loader order". eval_loader
    has shuffle=False, so "first N" was a fixed, position-dependent
    slice every time -- the same 300 images always got scored, and
    the resulting 0.00% clean risk needed checking on a genuine random
    draw rather than that one fixed slice.
    """
    out = {'confidence': [], 'disagreement': [], 'evidence': [], 'boundary': [],
           'correct_baseline': [], 'correct_multihead': [], 'correct_evidential': []}

    # First pass: figure out total size so we can draw a random boundary subset up front.
    total_available = n_samples if n_samples is not None else len(loader.dataset)
    rng = np.random.RandomState(boundary_seed)
    boundary_indices = set(rng.choice(total_available, size=min(n_boundary_samples, total_available),
                                       replace=False).tolist())

    n_seen = 0
    n_boundary_done = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if condition == 'adversarial':
            images = pgd_standard(baseline, images, labels, **PGD_PARAMS)

        with torch.no_grad():
            base_logits = baseline(images)
            base_probs = F.softmax(base_logits, dim=1)
            base_conf, base_pred = base_probs.max(dim=1)

            mean_probs, var_probs = multihead.forward_with_disagreement(images)
            mh_pred = mean_probs.argmax(dim=1)
            disagreement = var_probs.max(dim=1)[0]

            ev_probs, evidence = evidential.forward_with_evidence(images)
            ev_pred = ev_probs.argmax(dim=1)

        for i in range(images.size(0)):
            if n_samples is not None and n_seen >= n_samples:
                break
            out['confidence'].append(base_conf[i].item())
            out['disagreement'].append(disagreement[i].item())
            out['evidence'].append(evidence[i].item())
            out['correct_baseline'].append(int(base_pred[i].item() == labels[i].item()))
            out['correct_multihead'].append(int(mh_pred[i].item() == labels[i].item()))
            out['correct_evidential'].append(int(ev_pred[i].item() == labels[i].item()))

            if n_seen in boundary_indices:
                bd = estimate_boundary_distance(baseline, images[i:i+1], labels[i].item(), max_iters=30)
                out['boundary'].append(bd)
                n_boundary_done += 1
                if n_boundary_done % 100 == 0:
                    print(f"   ...boundary distance: {n_boundary_done}/{len(boundary_indices)}")
            else:
                out['boundary'].append(None)  # explicitly missing, not padded

            n_seen += 1
        if n_samples is not None and n_seen >= n_samples:
            break

    for k in out:
        out[k] = np.array(out[k], dtype=object) if k == 'boundary' else np.array(out[k])
    return out

print("Collecting calibration signals (clean + adversarial)...")
calib_clean = collect_signals(baseline_model, multi_head_model, evidential_model, calib_loader,
                               condition='clean', n_boundary_samples=400, boundary_seed=42)
calib_adv = collect_signals(baseline_model, multi_head_model, evidential_model, calib_loader,
                             condition='adversarial', n_boundary_samples=400, boundary_seed=43)

print("Collecting evaluation signals (clean + adversarial)...")
print("NOTE: boundary-distance sample raised to 1,200/condition and RANDOMIZED")
print("(previously 300, fixed to the first images in loader order).")
eval_clean = collect_signals(baseline_model, multi_head_model, evidential_model, eval_loader,
                              condition='clean', n_boundary_samples=1200, boundary_seed=44)
eval_adv = collect_signals(baseline_model, multi_head_model, evidential_model, eval_loader,
                            condition='adversarial', n_boundary_samples=1200, boundary_seed=45)


   ...boundary distance: 100/400
   ...boundary distance: 200/400
   ...boundary distance: 300/400
   ...boundary distance: 400/400
   ...boundary distance: 100/400
   ...boundary distance: 200/400
   ...boundary distance: 300/400
   ...boundary distance: 400/400
NOTE: boundary-distance sample raised to 1,200/condition and RANDOMIZED
(previously 300, fixed to the first images in loader order).
   ...boundary distance: 100/1200
   ...boundary distance: 200/1200
   ...boundary distance: 300/1200
   ...boundary distance: 400/1200
   ...boundary distance: 500/1200
   ...boundary distance: 600/1200
   ...boundary distance: 700/1200
   ...boundary distance: 800/1200
   ...boundary distance: 900/1200
   ...boundary distance: 1000/1200
   ...boundary distance: 1100/1200
   ...boundary distance: 1200/1200
   ...boundary distance: 100/1200
   ...boundary distance: 200/1200
   ...boundary distance: 300/1200
   ...boundary distance: 400/1200
   ...boundary distance: 500/1200
   ...boundary distanc

In [7]:
# -------------------------------------------------------------
# Part G: Calibrate thresholds ON CALIB DATA ONLY
# -------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.7  # matches Essays #4/#5's convention, kept fixed for comparability
disagreement_threshold = float(np.percentile(calib_clean['disagreement'], 75))
evidence_threshold = float(np.percentile(calib_clean['evidence'], 10))
boundary_vals_calib = np.array([b for b in calib_clean['boundary'] if b is not None])
boundary_threshold = float(np.percentile(boundary_vals_calib, 25))

print(f"\nThresholds calibrated on {len(calib_dataset)}-image calibration set:")
print(f"   Confidence:   {CONFIDENCE_THRESHOLD} (fixed, matches Essays 4/5)")
print(f"   Disagreement: {disagreement_threshold:.6f} (75th pct of clean calib)")
print(f"   Evidence:     {evidence_threshold:.4f} (10th pct of clean calib)")
print(f"   Boundary:     {boundary_threshold:.4f} (25th pct of clean calib)")



Thresholds calibrated on 2000-image calibration set:
   Confidence:   0.7 (fixed, matches Essays 4/5)
   Disagreement: 0.000000 (75th pct of clean calib)
   Evidence:     138.1205 (10th pct of clean calib)
   Boundary:     0.0030 (25th pct of clean calib)


In [8]:
# -------------------------------------------------------------
# Part H: Evaluate each signal as a standalone gate, Essay-4/5-style
# (coverage / accuracy-on-predicted / deferral / risk), including a
# STANDALONE boundary-distance row that was missing before.
# -------------------------------------------------------------

def gate_metrics(scores, correct, threshold, higher_is_fragile=True):
    scores = np.array([s for s in scores if s is not None], dtype=float)
    correct = correct[:len(scores)] if len(correct) != len(scores) else correct
    defer_mask = scores > threshold if higher_is_fragile else scores < threshold
    predict_mask = ~defer_mask
    n_total = len(scores)
    n_predicted = predict_mask.sum()
    n_deferred = defer_mask.sum()
    acc = 100 * correct[predict_mask].sum() / n_predicted if n_predicted > 0 else float('nan')
    return {
        'coverage': 100 * n_predicted / n_total,
        'accuracy': acc,
        'deferral_rate': 100 * n_deferred / n_total,
        'risk': 100 - acc if not np.isnan(acc) else float('nan'),  # FIXED: matches Essays 4/5
    }

def signal_report(name, clean_scores, clean_correct, adv_scores, adv_correct, threshold, higher_is_fragile=True):
    c = gate_metrics(clean_scores, clean_correct, threshold, higher_is_fragile)
    a = gate_metrics(adv_scores, adv_correct, threshold, higher_is_fragile)
    print(f"\n{name} (threshold={threshold:.4f}):")
    print(f"   Clean:       coverage={c['coverage']:.2f}%  accuracy={c['accuracy']:.2f}%  "
          f"deferral={c['deferral_rate']:.2f}%  risk={c['risk']:.2f}%")
    print(f"   Adversarial: coverage={a['coverage']:.2f}%  accuracy={a['accuracy']:.2f}%  "
          f"deferral={a['deferral_rate']:.2f}%  risk={a['risk']:.2f}%")
    return c, a

print("\n" + "=" * 60)
print("STANDALONE SIGNAL COMPARISON (calibrated on held-out split)")
print("=" * 60)

conf_clean, conf_adv = signal_report(
    "Confidence (baseline)", 1 - eval_clean['confidence'], eval_clean['correct_baseline'],
    1 - eval_adv['confidence'], eval_adv['correct_baseline'], 1 - CONFIDENCE_THRESHOLD, True)

disagree_clean, disagree_adv = signal_report(
    "Disagreement (multi-head)", eval_clean['disagreement'], eval_clean['correct_multihead'],
    eval_adv['disagreement'], eval_adv['correct_multihead'], disagreement_threshold, True)

evidence_clean, evidence_adv = signal_report(
    "Evidence (evidential)", eval_clean['evidence'], eval_clean['correct_evidential'],
    eval_adv['evidence'], eval_adv['correct_evidential'], evidence_threshold, False)  # LOW evidence = fragile

boundary_clean, boundary_adv = signal_report(
    "Boundary distance (standalone, baseline model)",
    eval_clean['boundary'], eval_clean['correct_baseline'],
    eval_adv['boundary'], eval_adv['correct_baseline'], boundary_threshold, False)  # LOW distance = fragile



STANDALONE SIGNAL COMPARISON (calibrated on held-out split)

Confidence (baseline) (threshold=0.3000):
   Clean:       coverage=70.19%  accuracy=92.45%  deferral=29.81%  risk=7.55%
   Adversarial: coverage=69.81%  accuracy=23.26%  deferral=30.19%  risk=76.74%

Disagreement (multi-head) (threshold=0.0000):
   Clean:       coverage=75.58%  accuracy=84.47%  deferral=24.43%  risk=15.53%
   Adversarial: coverage=71.86%  accuracy=78.76%  deferral=28.14%  risk=21.24%

Evidence (evidential) (threshold=138.1205):
   Clean:       coverage=90.61%  accuracy=79.65%  deferral=9.39%  risk=20.35%
   Adversarial: coverage=89.61%  accuracy=75.32%  deferral=10.39%  risk=24.68%

Boundary distance (standalone, baseline model) (threshold=0.0030):
   Clean:       coverage=73.25%  accuracy=79.07%  deferral=26.75%  risk=20.93%
   Adversarial: coverage=21.50%  accuracy=28.68%  deferral=78.50%  risk=71.32%


In [9]:
# -------------------------------------------------------------
# Part I: Fused gate -- OR-based (Essay #5 style), not a weighted
# blend of differently-scaled raw signals.
# -------------------------------------------------------------

def fused_or_gate_metrics(conf, disagreement, evidence, boundary, correct,
                           conf_thresh, dis_thresh, ev_thresh, bd_thresh, use_boundary=True):
    n = len(conf)
    boundary = np.array([b if b is not None else np.nan for b in boundary], dtype=float)
    defer = (conf < conf_thresh) | (disagreement > dis_thresh) | (evidence < ev_thresh)
    if use_boundary:
        has_boundary = ~np.isnan(boundary)
        defer = defer | (has_boundary & (boundary < bd_thresh))
    predict_mask = ~defer
    n_predicted = predict_mask.sum()
    acc = 100 * correct[predict_mask].sum() / n_predicted if n_predicted > 0 else float('nan')
    return {
        'coverage': 100 * n_predicted / n,
        'accuracy': acc,
        'deferral_rate': 100 * defer.sum() / n,
        'risk': 100 - acc if not np.isnan(acc) else float('nan'),
    }

# Use baseline's own correctness as ground truth for the fused gate,
# since it's the model actually making the final prediction.
fused_clean = fused_or_gate_metrics(
    eval_clean['confidence'], eval_clean['disagreement'], eval_clean['evidence'],
    eval_clean['boundary'], eval_clean['correct_baseline'],
    CONFIDENCE_THRESHOLD, disagreement_threshold, evidence_threshold, boundary_threshold
)
fused_adv = fused_or_gate_metrics(
    eval_adv['confidence'], eval_adv['disagreement'], eval_adv['evidence'],
    eval_adv['boundary'], eval_adv['correct_baseline'],
    CONFIDENCE_THRESHOLD, disagreement_threshold, evidence_threshold, boundary_threshold
)

print("\n" + "=" * 60)
print("FUSED GATE (OR-based: confidence OR disagreement OR evidence OR boundary)")
print("=" * 60)
print(f"Clean:       {fused_clean}")
print(f"Adversarial: {fused_adv}")

# For comparison: fused WITHOUT boundary distance, to isolate its contribution
fused_clean_nobd = fused_or_gate_metrics(
    eval_clean['confidence'], eval_clean['disagreement'], eval_clean['evidence'],
    eval_clean['boundary'], eval_clean['correct_baseline'],
    CONFIDENCE_THRESHOLD, disagreement_threshold, evidence_threshold, boundary_threshold, use_boundary=False
)
fused_adv_nobd = fused_or_gate_metrics(
    eval_adv['confidence'], eval_adv['disagreement'], eval_adv['evidence'],
    eval_adv['boundary'], eval_adv['correct_baseline'],
    CONFIDENCE_THRESHOLD, disagreement_threshold, evidence_threshold, boundary_threshold, use_boundary=False
)
print("\nFused gate WITHOUT boundary distance (confidence OR disagreement OR evidence only):")
print(f"Clean:       {fused_clean_nobd}")
print(f"Adversarial: {fused_adv_nobd}")
print("\nCompare the two fused rows above -- the gap between them is boundary")
print("distance's actual marginal contribution, which the previous weighted-sum")
print("design couldn't isolate.")



FUSED GATE (OR-based: confidence OR disagreement OR evidence OR boundary)
Clean:       {'coverage': np.float64(54.55), 'accuracy': np.float64(95.80659945004582), 'deferral_rate': np.float64(45.45), 'risk': np.float64(4.193400549954177)}
Adversarial: {'coverage': np.float64(40.625), 'accuracy': np.float64(37.323076923076925), 'deferral_rate': np.float64(59.375), 'risk': np.float64(62.676923076923075)}

Fused gate WITHOUT boundary distance (confidence OR disagreement OR evidence only):
Clean:       {'coverage': np.float64(54.9375), 'accuracy': np.float64(95.13083048919226), 'deferral_rate': np.float64(45.0625), 'risk': np.float64(4.869169510807737)}
Adversarial: {'coverage': np.float64(45.1), 'accuracy': np.float64(33.61973392461197), 'deferral_rate': np.float64(54.9), 'risk': np.float64(66.38026607538802)}

Compare the two fused rows above -- the gap between them is boundary
distance's actual marginal contribution, which the previous weighted-sum
design couldn't isolate.


In [10]:
# -------------------------------------------------------------
# Part J: Summary table -- all numbers use the SAME risk definition
# -------------------------------------------------------------

print("\n" + "=" * 70)
print(f"{'Method':<32}{'Clean cov':>10}{'Clean acc':>11}{'Clean risk':>12}"
      f"{'Adv cov':>10}{'Adv acc':>10}{'Adv risk':>10}")
print("-" * 70)
rows = [
    ("Confidence (baseline)", conf_clean, conf_adv),
    ("Disagreement (multi-head)", disagree_clean, disagree_adv),
    ("Evidence (evidential)", evidence_clean, evidence_adv),
    ("Boundary distance (standalone)", boundary_clean, boundary_adv),
    ("Fused (no boundary)", fused_clean_nobd, fused_adv_nobd),
    ("Fused (with boundary)", fused_clean, fused_adv),
]
for name, c, a in rows:
    print(f"{name:<32}{c['coverage']:>9.2f}%{c['accuracy']:>10.2f}%{c['risk']:>11.2f}%"
          f"{a['coverage']:>9.2f}%{a['accuracy']:>9.2f}%{a['risk']:>9.2f}%")

print("\n" + "!" * 70)
print("WARNING: the 'Boundary distance (standalone)' row above is evaluated")
print("on only the first 300 eval images (where boundary was actually")
print("computed), NOT the same 8,000-image population as every other row.")
print("Do not compare it directly to the rows above it. See Part K below")
print("for a fair, equal-sample comparison.")
print("!" * 70)



Method                           Clean cov  Clean acc  Clean risk   Adv cov   Adv acc  Adv risk
----------------------------------------------------------------------
Confidence (baseline)               70.19%     92.45%       7.55%    69.81%    23.26%    76.74%
Disagreement (multi-head)           75.58%     84.47%      15.53%    71.86%    78.76%    21.24%
Evidence (evidential)               90.61%     79.65%      20.35%    89.61%    75.32%    24.68%
Boundary distance (standalone)      73.25%     79.07%      20.93%    21.50%    28.68%    71.32%
Fused (no boundary)                 54.94%     95.13%       4.87%    45.10%    33.62%    66.38%
Fused (with boundary)               54.55%     95.81%       4.19%    40.62%    37.32%    62.68%

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
on only the first 300 eval images (where boundary was actually
computed), NOT the same 8,000-image population as every other row.
Do not compare it directly to the rows above it. See P

In [11]:
# -------------------------------------------------------------
# Part K: FAIR comparison -- all signals evaluated on the SAME
# subsample boundary distance was actually computed on, so the table
# is apples-to-apples. This is the comparison that should actually
# decide which direction "wins", not Part J's mismatched one.
# -------------------------------------------------------------

def restrict_to_boundary_sample(data_dict, n):
    """Return a copy of the signal dict restricted to the first n images
    -- the same slice that has real (non-None) boundary scores, since
    boundary distance was computed sequentially until its cap."""
    out = {}
    for k, v in data_dict.items():
        out[k] = v[:n]
    return out

N_BOUNDARY_CLEAN = sum(1 for b in eval_clean['boundary'] if b is not None)
N_BOUNDARY_ADV = sum(1 for b in eval_adv['boundary'] if b is not None)
print(f"\nRestricting fair comparison to {N_BOUNDARY_CLEAN} clean / {N_BOUNDARY_ADV} "
      f"adversarial images (the ones with a real boundary-distance score).")

fair_clean = restrict_to_boundary_sample(eval_clean, N_BOUNDARY_CLEAN)
fair_adv = restrict_to_boundary_sample(eval_adv, N_BOUNDARY_ADV)

print("\n" + "=" * 60)
print("FAIR STANDALONE SIGNAL COMPARISON (same sample for all four)")
print("=" * 60)

fair_conf_clean, fair_conf_adv = signal_report(
    "Confidence (baseline)", 1 - fair_clean['confidence'], fair_clean['correct_baseline'],
    1 - fair_adv['confidence'], fair_adv['correct_baseline'], 1 - CONFIDENCE_THRESHOLD, True)

fair_disagree_clean, fair_disagree_adv = signal_report(
    "Disagreement (multi-head)", fair_clean['disagreement'], fair_clean['correct_multihead'],
    fair_adv['disagreement'], fair_adv['correct_multihead'], disagreement_threshold, True)

fair_evidence_clean, fair_evidence_adv = signal_report(
    "Evidence (evidential)", fair_clean['evidence'], fair_clean['correct_evidential'],
    fair_adv['evidence'], fair_adv['correct_evidential'], evidence_threshold, False)

fair_boundary_clean, fair_boundary_adv = signal_report(
    "Boundary distance", fair_clean['boundary'], fair_clean['correct_baseline'],
    fair_adv['boundary'], fair_adv['correct_baseline'], boundary_threshold, False)

print("\n" + "=" * 70)
print(f"{'Method (fair, n=' + str(N_BOUNDARY_CLEAN) + '/' + str(N_BOUNDARY_ADV) + ')':<32}"
      f"{'Clean cov':>10}{'Clean acc':>11}{'Clean risk':>12}"
      f"{'Adv cov':>10}{'Adv acc':>10}{'Adv risk':>10}")
print("-" * 70)
fair_rows = [
    ("Confidence (baseline)", fair_conf_clean, fair_conf_adv),
    ("Disagreement (multi-head)", fair_disagree_clean, fair_disagree_adv),
    ("Evidence (evidential)", fair_evidence_clean, fair_evidence_adv),
    ("Boundary distance", fair_boundary_clean, fair_boundary_adv),
]
for name, c, a in fair_rows:
    print(f"{name:<32}{c['coverage']:>9.2f}%{c['accuracy']:>10.2f}%{c['risk']:>11.2f}%"
          f"{a['coverage']:>9.2f}%{a['accuracy']:>9.2f}%{a['risk']:>9.2f}%")
print("\nThis table, not Part J's, is the one that should decide which signal wins --")
print("every row here is judged against the same population.")



Restricting fair comparison to 1200 clean / 1200 adversarial images (the ones with a real boundary-distance score).

FAIR STANDALONE SIGNAL COMPARISON (same sample for all four)

Confidence (baseline) (threshold=0.3000):
   Clean:       coverage=69.42%  accuracy=91.72%  deferral=30.58%  risk=8.28%
   Adversarial: coverage=70.92%  accuracy=24.32%  deferral=29.08%  risk=75.68%

Disagreement (multi-head) (threshold=0.0000):
   Clean:       coverage=75.92%  accuracy=83.86%  deferral=24.08%  risk=16.14%
   Adversarial: coverage=72.25%  accuracy=78.66%  deferral=27.75%  risk=21.34%

Evidence (evidential) (threshold=138.1205):
   Clean:       coverage=90.08%  accuracy=78.35%  deferral=9.92%  risk=21.65%
   Adversarial: coverage=89.25%  accuracy=74.04%  deferral=10.75%  risk=25.96%

Boundary distance (threshold=0.0030):
   Clean:       coverage=73.37%  accuracy=85.48%  deferral=26.63%  risk=14.52%
   Adversarial: coverage=20.99%  accuracy=28.95%  deferral=79.01%  risk=71.05%

Method (fair, n=

In [12]:
# -------------------------------------------------------------
# Part L: Coverage-matched comparison. Part K's table gives each
# signal ITS OWN calibrated operating point, but those points aren't
# at matched coverage -- boundary distance defers far more often
# (much lower coverage) than the other three, so part of its risk
# advantage could simply be conservatism, not a sharper signal. This
# finds, for each of confidence/disagreement/evidence, the threshold
# that reproduces boundary distance's OWN adversarial coverage, then
# reports risk at that matched point. Thresholds are searched on the
# CALIBRATION set (not fair_adv itself) to keep the held-out
# methodology intact.
# -------------------------------------------------------------

def find_threshold_for_coverage(calib_scores, target_coverage_pct, higher_is_fragile=True):
    """Binary-search a threshold on calibration scores so that
    deferral rate matches (100 - target_coverage_pct)."""
    target_deferral_pct = 100 - target_coverage_pct
    lo, hi = float(np.min(calib_scores)), float(np.max(calib_scores))
    for _ in range(50):
        mid = (lo + hi) / 2
        defer_pct = 100 * (calib_scores > mid).mean() if higher_is_fragile else 100 * (calib_scores < mid).mean()
        if defer_pct > target_deferral_pct:
            if higher_is_fragile:
                lo = mid
            else:
                hi = mid
        else:
            if higher_is_fragile:
                hi = mid
            else:
                lo = mid
    return (lo + hi) / 2

target_coverage = fair_boundary_adv['coverage']
print(f"\nTarget adversarial coverage (boundary distance's own operating point): {target_coverage:.2f}%")

conf_matched_thresh = find_threshold_for_coverage(1 - calib_adv['confidence'], target_coverage, True)
disagree_matched_thresh = find_threshold_for_coverage(calib_adv['disagreement'], target_coverage, True)
evidence_matched_thresh = find_threshold_for_coverage(calib_adv['evidence'], target_coverage, False)

print(f"Coverage-matched thresholds (calibrated on calib_adv):")
print(f"   Confidence:   {conf_matched_thresh:.4f}  (vs. own threshold {1 - CONFIDENCE_THRESHOLD:.4f})")
print(f"   Disagreement: {disagree_matched_thresh:.6f}  (vs. own threshold {disagreement_threshold:.6f})")
print(f"   Evidence:     {evidence_matched_thresh:.4f}  (vs. own threshold {evidence_threshold:.4f})")

matched_conf_adv = gate_metrics(1 - fair_adv['confidence'], fair_adv['correct_baseline'], conf_matched_thresh, True)
matched_disagree_adv = gate_metrics(fair_adv['disagreement'], fair_adv['correct_multihead'], disagree_matched_thresh, True)
matched_evidence_adv = gate_metrics(fair_adv['evidence'], fair_adv['correct_evidential'], evidence_matched_thresh, False)

print("\n" + "=" * 70)
print(f"{'Method (adversarial, coverage-matched to ~' + f'{target_coverage:.0f}%)':<45}"
      f"{'Coverage':>10}{'Accuracy':>10}{'Risk':>10}")
print("-" * 70)
for name, m in [
    ("Confidence (own threshold)", fair_conf_adv),
    ("Confidence (coverage-matched)", matched_conf_adv),
    ("Disagreement (own threshold)", fair_disagree_adv),
    ("Disagreement (coverage-matched)", matched_disagree_adv),
    ("Evidence (own threshold)", fair_evidence_adv),
    ("Evidence (coverage-matched)", matched_evidence_adv),
    ("Boundary distance (own threshold)", fair_boundary_adv),
]:
    print(f"{name:<45}{m['coverage']:>9.2f}%{m['accuracy']:>9.2f}%{m['risk']:>9.2f}%")

print("\nIf the coverage-matched rows still show meaningfully higher risk than boundary")
print("distance at the SAME coverage, that's real evidence boundary distance carries")
print("more information, not just a more conservative threshold. If they converge,")
print("most of boundary distance's apparent edge was coverage, not information.")



Target adversarial coverage (boundary distance's own operating point): 20.99%
Coverage-matched thresholds (calibrated on calib_adv):
   Confidence:   0.0214  (vs. own threshold 0.3000)
   Disagreement: 0.000000  (vs. own threshold 0.000000)
   Evidence:     625.8625  (vs. own threshold 138.1205)

Method (adversarial, coverage-matched to ~21%)  Coverage  Accuracy      Risk
----------------------------------------------------------------------
Confidence (own threshold)                       70.92%    24.32%    75.68%
Confidence (coverage-matched)                    21.92%    20.91%    79.09%
Disagreement (own threshold)                     72.25%    78.66%    21.34%
Disagreement (coverage-matched)                  22.67%    98.53%     1.47%
Evidence (own threshold)                         89.25%    74.04%    25.96%
Evidence (coverage-matched)                      24.17%    92.07%     7.93%
Boundary distance (own threshold)                20.99%    28.95%    71.05%

If the coverage-matc